# NyayaAI - Colab Training Notebook

End-to-end pipeline for **IndiaLex-FT** — a LoRA fine-tuned LLM for Indian legal and income-tax Q&A.

**Steps covered:**
1. Clone repo and install dependencies
2. Load API keys from Colab Secrets
3. HuggingFace login
4. Generate training data
5. Baseline evaluation (pre-training)
6. LoRA SFT fine-tuning
7. Post-training evaluation
8. LLM-as-Judge scoring via claude-sonnet-4-6
9. Download outputs

> **Before running:** Select `Runtime → Change runtime type → T4 GPU`
> Then add your secrets under the 🔑 icon in the left sidebar:
> `ANTHROPIC_API_KEY`, `GROQ_API_KEY`, `HF_TOKEN` (required), `WANDB_API_KEY` (optional)

In [ ]:
!git clone https://github.com/Harshith2014/Nyaya-AI.git
%cd Nyaya-AI/indialex-ft

In [2]:
!pip install -r requirements.txt --quiet

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
import os
from google.colab import userdata

def _load_secret(name, required=True):
    try:
        val = userdata.get(name)
        if not val:
            raise ValueError('empty')
        os.environ[name] = val
        print(f'{name} loaded: True')
    except Exception:
        if required:
            raise EnvironmentError(f'{name} is missing — add it in the 🔑 Secrets panel.')
        else:
            print(f'{name} loaded: skipped (optional)')

_load_secret('ANTHROPIC_API_KEY', required=True)
_load_secret('GROQ_API_KEY',      required=True)
_load_secret('HF_TOKEN',          required=True)
_load_secret('WANDB_API_KEY',     required=False)

In [ ]:
import os
from huggingface_hub import login

hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    login(token=hf_token)
    print('HuggingFace login successful.')
else:
    raise EnvironmentError(
        'HF_TOKEN is not set. Add it in the 🔑 Secrets panel '
        'and enable Notebook access.'
    )

## Step 0: Generate Training Data

Generates synthetic Indian legal Q&A pairs via Groq API and splits them
into train / val / test sets under `data/splits/`.
Adjust `--total` to control how many pairs are generated (default 500).

In [ ]:
!python scripts/data_pipeline.py --synthetic --total 500

## Step 1: Baseline Evaluation

Runs inference on the test split using the **base model** (before fine-tuning).
Computes ROUGE-L and BERTScore F1, and saves results to `evals/baseline_results.json`.

In [ ]:
!python scripts/baseline_eval.py

## Step 2: Training

Fine-tunes the base model with **LoRA + QLoRA (4-bit NF4)** via TRL SFTTrainer.
Saves the LoRA adapter to `outputs/adapter/` and the merged model to `outputs/merged/`.
Logs metrics to Weights & Biases.

In [ ]:
!python scripts/train.py

## Step 3: Post-training Evaluation

Runs inference on the test split using the **fine-tuned merged model**.
Computes ROUGE-L and BERTScore F1, compares against baseline, and saves
a delta summary to `evals/ft_results.json`.

In [ ]:
!python scripts/evaluate.py

## Step 4: LLM-as-Judge Evaluation

Uses **claude-sonnet-4-6** (Anthropic API) to score each fine-tuned answer on
four legal-domain dimensions (1–5 each): correctness, faithfulness, helpfulness,
and legal accuracy. Results saved to `evals/judge_results.json`.

In [ ]:
!python evals/llm_judge.py --results evals/ft_results.json

In [ ]:
import shutil
from pathlib import Path

for name, base_dir in [("nyayaai_outputs", "outputs"), ("nyayaai_evals", "evals")]:
    if Path(base_dir).exists():
        shutil.make_archive(name, "zip", root_dir=".", base_dir=base_dir)
        print(f"Zipped {base_dir}/ → {name}.zip")
    else:
        print(f"Skipped {base_dir}/ — directory does not exist yet.")

try:
    from google.colab import files
    for f in ["nyayaai_outputs.zip", "nyayaai_evals.zip"]:
        if Path(f).exists():
            files.download(f)
except ImportError:
    print("Zip files saved locally (not running in Colab).")